<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/11-catalyst-optimizer/03-whole-stage-codegen.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 11.3 — Whole-stage codegen: what the `*` means

Find the `*(n)` markers, read the generated Java, then watch a Python
UDF split a single compiled loop into interpreted pieces with a
Python round-trip in the middle. The `*` is the difference between a
tight loop and the volcano tax.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("11.3-whole-stage-codegen")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
df = spark.range(5_000_000).withColumn("k", (col("id") % 50).cast("int")) \
                             .withColumn("v", col("id") * 2)

## The `*(n)` prefix: operators fused into one compiled function

Scan+filter+project+partial-agg share `*(1)` — one Java loop. The
`Exchange` breaks fusion; the final agg is `*(2)`. Codegen units ≈
stages.

In [ ]:
df.filter(col("v") > 100).groupBy("k").sum("v").explain()
# read the * numbers: everything below the Exchange is *(1), above is *(2)

## The generated Java, for real

`codegen` mode dumps the compiled source. Skim for the `while`/`for`
loop and the inlined comparison — no per-operator virtual calls.

In [ ]:
df.filter(col("v") > 100).select("k").explain(mode="codegen")

## A Python UDF poisons the loop

An `EvalPython` node appears with **no** `*` and splits the codegen
unit: rows leave the JVM, cross to a Python worker, come back. The
visual signature of 9.0's "three tolls" in a plan.

In [ ]:
from pyspark.sql.types import LongType

@F.udf(LongType())
def plus_one(x):
    return x + 1

print("NATIVE — one fused *(1) loop:")
df.select((col("v") + 1).alias("v1")).explain()

print("\nUDF — un-starred EvalPython splits the codegen unit:")
df.select(plus_one(col("v")).alias("v1")).explain()

## Measure the tax

Same arithmetic (`+1`), two execution models. The UDF pays
serialization + interpreter on every row (Module 9); native stays in
the compiled loop.

In [ ]:
import time
def timed(df, label):
    t = time.time(); df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.time()-t:.2f}s")

timed(df.select((col("v") + 1).alias("v1")),        "native (compiled) ")
timed(df.select(plus_one(col("v")).alias("v1")),    "python udf        ")

## Codegen is on by default — you keep it from turning itself off

Prove the switch exists (don't run production with it off — this is
only to see the volcano model it replaces).

In [ ]:
print("wholeStage codegen:", spark.conf.get("spark.sql.codegen.wholeStage"))

spark.conf.set("spark.sql.codegen.wholeStage", "false")
print("\nWith codegen OFF — the * markers vanish (interpreted volcano):")
df.filter(col("v") > 100).select("k").explain()
spark.conf.set("spark.sql.codegen.wholeStage", "true")   # restore

In [ ]:
spark.stop()

## Exercises

1. In the groupBy plan, list which operators share `*(1)` and which
   share `*(2)`. What node sits exactly on the boundary, and why can't
   codegen fuse across it?
2. Add a second UDF after the first: `select(plus_one(plus_one(v)))`.
   How many `EvalPython` nodes appear? Do they merge or stack?
3. Turn codegen off (as above) and time the native `+1` query. Compare
   to codegen-on. Roughly how much does the volcano model cost here?
4. Build a very wide projection — 300 `withColumn` calls of small
   expressions — and check for a codegen fallback warning (the 64 KB
   method limit). Does the plan still show `*`?
5. Replace the UDF with a Pandas UDF (Module 9.1) doing `+1`. Does the
   plan show `ArrowEvalPython` instead of `BatchEvalPython`? Re-time
   it — where does it land between native and the plain Python UDF?

In [ ]:
# your exercise code here